In [1]:
# ============================================================================
# 06_generation_and_delta.ipynb
# LLM experiment, part 1 (requires an OpenAI key in ../.env). Generate the
# explanation corpus, extract structured claims with the validated extractor,
# measure the content distance between explanations of the same case, and
# attribute that variation to a pipeline stage (the main delta analysis,
# RQ1-RQ3), with a mixed-effects robustness check.
#
# Three generators are read from the environment and mapped to capability tiers:
#   LLM_MODEL_WEAK   -> weak      (budget)
#   LLM_MODEL_MID    -> reference (held fixed when the generator is not varied)
#   LLM_MODEL_STRONG -> strong    (highest capability)
# Every API call is cached on disk by a key over all inputs, so an interrupted
# run resumes without re-issuing completed calls and a repeat run regenerates
# nothing. No file path is ever printed (author anonymity).
# ============================================================================

# [Imports and global configuration]
import os
import re
import json
import hashlib
import warnings
from pathlib import Path
from itertools import combinations
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kendalltau
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

ROOT = Path("..")
DATA_DIR = ROOT / "data"
PROMPT_DIR = ROOT / "prompts"
GEN_DIR = DATA_DIR / "generations"
CLAIM_DIR = DATA_DIR / "claims_cache"
FIG_DIR = ROOT / "results" / "figures"
TAB_DIR = ROOT / "results" / "tables"
for d in (GEN_DIR, CLAIM_DIR, FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Credentials come from ../.env (a sibling of the notebooks folder); only a
# neutral confirmation is printed, never the path.
_env = find_dotenv(usecwd=True)
if _env:
    load_dotenv(_env)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found; add it to ../.env before running.")
client = OpenAI()
print("credentials loaded")

# [Plotting style: grayscale, publication settings, no captions]
sns.set_theme(style="whitegrid", context="paper")
sns.set_palette(sns.color_palette("gray", n_colors=8))
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "font.family": "sans-serif",
    "axes.grid": True, "grid.color": "0.85", "grid.linewidth": 0.5,
    "axes.edgecolor": "0.3", "axes.labelcolor": "0.15", "text.color": "0.15",
    "xtick.color": "0.3", "ytick.color": "0.3", "image.cmap": "gray",
    "savefig.bbox": "tight", "savefig.transparent": False,
})

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
    print(f"saved figure: {name}")

def save_table(df, name, index=False):
    df.to_csv(TAB_DIR / f"{name}.csv", index=index, encoding="utf-8-sig")
    print(f"saved table: {name}")

# [Configuration: generators, prompts, blocks]
GENERATORS = {"weak": os.getenv("LLM_MODEL_WEAK", "gpt-4o-mini"),
              "reference": os.getenv("LLM_MODEL_MID", "gpt-5.4-mini"),
              "strong": os.getenv("LLM_MODEL_STRONG", "gpt-5.4")}
REFERENCE_GENERATOR = "reference"
REFERENCE_MODEL, REFERENCE_XAI, REFERENCE_PROMPT = "XGBoost", "SHAP", "basic"
PROMPTS = {name: (PROMPT_DIR / f"{name}_v1.txt").read_text(encoding="utf-8")
           for name in ("basic", "constrained", "verification")}
TOP_K = 5
N_REPEATS_BASELINE = 3          # Block D regenerations under identical settings
MAX_TOKENS = 400
RANK_PRIMARY = True
B_BOOTSTRAP, CI_LEVEL = 2000, 0.95

with open(DATA_DIR / "feature_registry.json", encoding="utf-8") as f:
    registry = json.load(f)
with open(DATA_DIR / "validation_summary.json", encoding="utf-8") as f:
    validation = json.load(f)
if not validation.get("proceed", False):
    raise RuntimeError("Extractor gates not met in notebook 04; fix before generating.")
with open(DATA_DIR / "extractor_config.json", encoding="utf-8") as f:
    extractor_config = json.load(f)
EXTRACTOR_MODEL = extractor_config["extractor_model"]
EXTRACTOR_SYSTEM = extractor_config["extractor_system_prompt"]

evidence = pd.read_json(DATA_DIR / "evidence_blocks.json")
evidence["key"] = (evidence["domain"] + "|" + evidence["case_id"].astype(str) + "|"
                   + evidence["model"] + "|" + evidence["xai"])
ev_lookup = {k: d for k, d in zip(evidence["key"], evidence["drivers"])}
ev_meta = {k: (bv, pp) for k, bv, pp in
           zip(evidence["key"], evidence["base_value"], evidence["predicted_probability"])}
# Case-level adverse-decision flag (the "rescue target" subgroup), carried from
# notebook 03 so every analysis can be reported for the full sample AND for the
# adverse subgroup where that concept applies.
adverse_by_case = {(d, str(c)): bool(a) for d, c, a in
                   zip(evidence["domain"], evidence["case_id"], evidence.get("adverse", False))}
adverse_applicable = {d: bool(x) for d, x in
                      zip(evidence["domain"], evidence.get("adverse_applicable", False))}
print(f"evidence blocks: {len(evidence):,}")

# [Prompt construction and cached LLM calls]
def humanise(feat):
    # Turn a one-hot identifier into a short readable phrase for the narrative.
    return feat.replace("_", " ").strip()

def evidence_text(drivers, base_value, pred_prob):
    verdict = "above" if pred_prob >= base_value else "below"
    lines = [f"Overall, this case scores {verdict} the typical level "
             f"(assessment {pred_prob:.2f}, typical {base_value:.2f})."]
    lines.append("Factors, most important first:")
    for d in drivers[:TOP_K]:
        updown = "raised" if d["direction"] == "increases" else "lowered"
        lines.append(f"  {d['rank']}. {humanise(d['feature'])} -- {updown} the assessment")
    return "\n".join(lines)

def cache_key(*parts):
    return hashlib.sha1("||".join(map(str, parts)).encode()).hexdigest()

def chat(model, system, user, seed=SEED, cache_dir=GEN_DIR, max_tokens=MAX_TOKENS):
    # Cached chat call. gpt-5.x models accept only default temperature, so we do
    # not send one; seed is best-effort and dropped if the model rejects it.
    key = cache_key(model, system, user, seed, max_tokens)
    fp = cache_dir / f"{key}.json"
    if fp.exists():
        return json.loads(fp.read_text(encoding="utf-8"))["text"]
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    kwargs = dict(model=model, messages=msgs, max_completion_tokens=max_tokens, seed=seed)
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(**kwargs)
            text = resp.choices[0].message.content or ""
            fp.write_text(json.dumps({"text": text}), encoding="utf-8")
            return text
        except TypeError:
            kwargs.pop("seed", None); kwargs["max_tokens"] = kwargs.pop("max_completion_tokens", max_tokens)
        except Exception as exc:
            if attempt == 2:
                raise
    return ""

def generate(domain, case_id, model, xai, generator, prompt_name, seed=SEED):
    k = f"{domain}|{case_id}|{model}|{xai}"
    drivers = ev_lookup[k]; bv, pp = ev_meta[k]
    user = (f"EVIDENCE\n{evidence_text(drivers, bv, pp)}\n\n"
            f"Write the explanation now.")
    return chat(GENERATORS[generator], PROMPTS[prompt_name], user, seed=seed)

# [Enumerate the experimental conditions as (block, factor) contrasts]
# Block A : evidence stage  -> vary model and xai; generator/prompt fixed.
# Block B : generation stage-> vary generator and prompt; evidence fixed.
# Block D : repeat baseline -> reference condition regenerated N_REPEATS times.
cases = (evidence[["domain", "case_id"]].drop_duplicates()
         .groupby("domain")["case_id"].apply(list).to_dict())
models_by_domain = evidence.groupby("domain")["model"].unique().apply(list).to_dict()

conditions = []   # each: dict(domain, case_id, model, xai, generator, prompt, rep)
def add(domain, cid, model, xai, gen, prompt, rep=0):
    conditions.append(dict(domain=domain, case_id=cid, model=model, xai=xai,
                           generator=gen, prompt=prompt, rep=rep))
for domain, cids in cases.items():
    ms = models_by_domain[domain]
    for cid in cids:
        # Block A: evidence stage (reference generator + prompt).
        for model in ms:
            for xai in ("SHAP", "LIME"):
                add(domain, cid, model, xai, REFERENCE_GENERATOR, REFERENCE_PROMPT)
        # Block B: generation stage (reference evidence).
        for gen in GENERATORS:
            for prompt in PROMPTS:
                add(domain, cid, REFERENCE_MODEL, REFERENCE_XAI, gen, prompt)
        # Block D: repeat baseline (reference everything), extra regenerations.
        for rep in range(1, N_REPEATS_BASELINE):
            add(domain, cid, REFERENCE_MODEL, REFERENCE_XAI, REFERENCE_GENERATOR,
                REFERENCE_PROMPT, rep=rep)
cond_df = pd.DataFrame(conditions).drop_duplicates(
    subset=["domain", "case_id", "model", "xai", "generator", "prompt", "rep"]).reset_index(drop=True)
print(f"conditions to generate: {len(cond_df):,}")

# [Generate the corpus (threaded, cached)]
def run_generation(i):
    r = cond_df.loc[i]
    seed = SEED + int(r["rep"])
    text = generate(r["domain"], r["case_id"], r["model"], r["xai"],
                    r["generator"], r["prompt"], seed=seed)
    return i, text

texts = [None] * len(cond_df)
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(run_generation, i) for i in range(len(cond_df))]
    done = 0
    for fut in as_completed(futs):
        i, text = fut.result(); texts[i] = text; done += 1
        if done % 500 == 0:
            print(f"generated {done:,}/{len(cond_df):,}")
cond_df["explanation"] = texts
cond_df.to_parquet(DATA_DIR / "generation_corpus.parquet", index=False)
print(f"corpus complete: {len(cond_df):,} explanations")

# [Extract structured claims with the validated extractor (cached)]
def extract_claims(domain, explanation):
    allowed = registry[domain]["features"]
    user = ("ALLOWED FEATURES\n" + ", ".join(allowed) +
            "\n\nEXPLANATION\n" + explanation)
    raw = chat(EXTRACTOR_MODEL, EXTRACTOR_SYSTEM, user, cache_dir=CLAIM_DIR, max_tokens=700)
    try:
        data = json.loads(re.sub(r"^```[a-z]*|```$", "", raw.strip(), flags=re.M))
        return [c for c in data if isinstance(c, dict) and c.get("feature") not in (None, "OUT_OF_LIST")]
    except Exception:
        return []

def run_extract(i):
    r = cond_df.loc[i]
    return i, extract_claims(r["domain"], r["explanation"])

claims = [None] * len(cond_df)
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(run_extract, i) for i in range(len(cond_df))]
    done = 0
    for fut in as_completed(futs):
        i, cl = fut.result(); claims[i] = cl; done += 1
        if done % 500 == 0:
            print(f"extracted {done:,}/{len(cond_df):,}")
cond_df["claims"] = claims
cond_df.to_parquet(DATA_DIR / "claims_corpus.parquet", index=False)

# [Content distance between two explanations of the same case]
def feat_set(cl):
    return frozenset(c["feature"] for c in cl)
def dir_map(cl):
    return {c["feature"]: c["direction"] for c in cl}
def rank_map(cl):
    return {c["feature"]: c.get("rank", i + 1) for i, c in enumerate(cl)}

def distances(cl_a, cl_b):
    a, b = feat_set(cl_a), feat_set(cl_b)
    union = a | b
    feat_d = 1 - (len(a & b) / len(union) if union else 1.0)
    shared = a & b
    da, db = dir_map(cl_a), dir_map(cl_b)
    dir_d = (1 - np.mean([da[f] == db[f] for f in shared])) if shared else (1.0 if union else 0.0)
    if len(shared) >= 2:
        ra = [rank_map(cl_a)[f] for f in shared]; rb = [rank_map(cl_b)[f] for f in shared]
        tau = kendalltau(ra, rb).correlation
        rank_d = (1 - tau) / 2 if tau == tau else 0.0
    else:
        rank_d = 0.0
    return feat_d, dir_d, rank_d

idx = {}
for i, r in cond_df.iterrows():
    idx[(r["domain"], r["case_id"], r["model"], r["xai"], r["generator"], r["prompt"], r["rep"])] = i

def claims_at(domain, cid, model, xai, gen, prompt, rep=0):
    i = idx.get((domain, cid, model, xai, gen, prompt, rep))
    return cond_df.loc[i, "claims"] if i is not None else None

# [Case-level deltas per factor, relative to the repeat baseline]
def repeat_baseline(domain, cid):
    # Mean pairwise distance across the reference regenerations (Block D).
    reps = [claims_at(domain, cid, REFERENCE_MODEL, REFERENCE_XAI,
                      REFERENCE_GENERATOR, REFERENCE_PROMPT, rep=r)
            for r in range(N_REPEATS_BASELINE)]
    reps = [c for c in reps if c is not None]
    ds = [distances(a, b) for a, b in combinations(reps, 2)]
    return np.mean(ds, axis=0) if ds else np.array([np.nan, np.nan, np.nan])

FACTORS = {
    "XAI method": lambda d, c: (claims_at(d, c, REFERENCE_MODEL, "SHAP", REFERENCE_GENERATOR, REFERENCE_PROMPT),
                                claims_at(d, c, REFERENCE_MODEL, "LIME", REFERENCE_GENERATOR, REFERENCE_PROMPT)),
    "Prediction model": lambda d, c: (claims_at(d, c, "XGBoost", REFERENCE_XAI, REFERENCE_GENERATOR, REFERENCE_PROMPT),
                                      claims_at(d, c, "LightGBM", REFERENCE_XAI, REFERENCE_GENERATOR, REFERENCE_PROMPT)),
    "LLM": lambda d, c: (claims_at(d, c, REFERENCE_MODEL, REFERENCE_XAI, "reference", REFERENCE_PROMPT),
                         claims_at(d, c, REFERENCE_MODEL, REFERENCE_XAI, "strong", REFERENCE_PROMPT)),
    "Prompt": lambda d, c: (claims_at(d, c, REFERENCE_MODEL, REFERENCE_XAI, REFERENCE_GENERATOR, "basic"),
                           claims_at(d, c, REFERENCE_MODEL, REFERENCE_XAI, REFERENCE_GENERATOR, "constrained")),
}
STAGE = {"XAI method": "Evidence", "Prediction model": "Evidence", "LLM": "Generation", "Prompt": "Generation"}
METRICS = ["feature", "direction", "rank"]

rows = []
for domain, cids in cases.items():
    for cid in cids:
        base = repeat_baseline(domain, cid)
        for factor, getter in FACTORS.items():
            a, b = getter(domain, cid)
            if a is None or b is None:
                continue
            d = distances(a, b)
            rows.append(dict(domain=domain, case_id=cid, factor=factor, stage=STAGE[factor],
                             adverse=adverse_by_case.get((domain, str(cid)), False),
                             base_feature=base[0], base_direction=base[1], base_rank=base[2],
                             feature=d[0], direction=d[1], rank=d[2]))
delta_df = pd.DataFrame(rows)
for m in METRICS:
    delta_df[f"delta_{m}"] = delta_df[m] - delta_df[f"base_{m}"]
delta_df.to_parquet(DATA_DIR / "delta_case_level.parquet", index=False)

def cluster_bootstrap(df, col, B=B_BOOTSTRAP, level=CI_LEVEL):
    cases_arr = df["case_id"].values; vals = df[col].values
    uniq = np.unique(cases_arr)
    rng = np.random.RandomState(SEED)
    means = np.empty(B)
    by_case = {c: vals[cases_arr == c] for c in uniq}
    for b in range(B):
        pick = rng.choice(uniq, len(uniq), replace=True)
        means[b] = np.concatenate([by_case[c] for c in pick]).mean()
    lo, hi = np.percentile(means, [(1 - level) / 2 * 100, (1 + level) / 2 * 100])
    return float(vals.mean()), float(lo), float(hi)

# [Main delta table: primary domain, feature distance; full sample + adverse]
prim = next(d for d in registry if registry[d]["primary"])
MIN_SUBGROUP = 20
main_rows = []
subgroups = [("all", delta_df[delta_df.domain == prim])]
if adverse_applicable.get(prim, False):
    subgroups.append(("adverse", delta_df[(delta_df.domain == prim) & (delta_df.adverse)]))
for sgname, sgdf in subgroups:
    for factor in FACTORS:
        sub = sgdf[sgdf.factor == factor]
        if sub["case_id"].nunique() < MIN_SUBGROUP:
            continue
        dmean, lo, hi = cluster_bootstrap(sub, "delta_feature")
        main_rows.append({"subgroup": sgname, "stage": STAGE[factor], "factor": factor,
                          "baseline": round(sub["base_feature"].mean(), 3),
                          "changed": round(sub["feature"].mean(), 3),
                          "delta": round(dmean, 3), "ci_low": round(lo, 3), "ci_high": round(hi, 3)})
main = pd.DataFrame(main_rows)
save_table(main, "tab39_delta_main_results")
print(main.to_string(index=False))
main_all = main[main.subgroup == "all"]  # used for the figures below

# [Figure 27: delta by factor (primary domain, feature distance)]
order = main_all.sort_values("delta", ascending=True)
fig, ax = plt.subplots(figsize=(6.6, 3.6))
y = np.arange(len(order))
shades = ["0.25" if s == "Evidence" else "0.6" for s in order["stage"]]
ax.barh(y, order["delta"], color=shades, edgecolor="0.15",
        xerr=[order["delta"] - order["ci_low"], order["ci_high"] - order["delta"]],
        error_kw=dict(ecolor="0.1", lw=0.8, capsize=2))
ax.set_yticks(y); ax.set_yticklabels(order["factor"], fontsize=8)
ax.set_xlabel("Additional variation (delta) over repeat baseline")
save_fig(fig, "fig27_delta_by_factor")

# [Figure 28: baseline vs changed distance by factor]
fig, ax = plt.subplots(figsize=(6.8, 3.6))
x = np.arange(len(main_all)); w = 0.38
ax.bar(x - w / 2, main_all["baseline"], width=w, color="0.7", edgecolor="0.15", label="repeat baseline")
ax.bar(x + w / 2, main_all["changed"], width=w, color="0.35", edgecolor="0.15", label="changed condition")
ax.set_xticks(x); ax.set_xticklabels(main_all["factor"], rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Feature distance"); ax.legend(fontsize=7, frameon=False)
save_fig(fig, "fig28_baseline_vs_changed")

# [Figure 29: full sample vs adverse-decision subgroup, delta by factor]
if (main["subgroup"] == "adverse").any():
    piv = main.pivot_table(index="factor", columns="subgroup", values="delta").reindex(list(FACTORS))
    fig, ax = plt.subplots(figsize=(6.8, 3.6))
    x = np.arange(len(piv)); w = 0.38
    ax.bar(x - w / 2, piv["all"], width=w, color="0.55", edgecolor="0.15", label="full sample")
    ax.bar(x + w / 2, piv["adverse"], width=w, color="0.25", edgecolor="0.15", label="adverse subgroup")
    ax.set_xticks(x); ax.set_xticklabels(piv.index, rotation=20, ha="right", fontsize=8)
    ax.set_ylabel("Delta (feature distance)"); ax.legend(fontsize=7, frameon=False)
    save_fig(fig, "fig29_delta_full_vs_adverse")

# [Figure 30: cumulative distribution of case-level delta by factor]
fig, ax = plt.subplots(figsize=(6.0, 3.8))
styles = {"XAI method": "-", "Prediction model": "-.", "LLM": "--", "Prompt": ":"}
for factor in FACTORS:
    v = np.sort(delta_df[(delta_df.domain == prim) & (delta_df.factor == factor)]["delta_feature"].values)
    if len(v):
        ax.plot(v, np.linspace(0, 1, len(v)), linestyle=styles[factor],
                color="0.2" if STAGE[factor] == "Evidence" else "0.55", lw=1.3, label=factor)
ax.set_xlabel("Case-level delta (feature distance)"); ax.set_ylabel("Cumulative fraction")
ax.legend(fontsize=7, frameon=False, loc="lower right")
save_fig(fig, "fig30_delta_cumulative_distribution")

# [Delta across all domains, full sample and adverse subgroup]
all_rows = []
for domain in registry:
    sgs = [("all", delta_df[delta_df.domain == domain])]
    if adverse_applicable.get(domain, False):
        sgs.append(("adverse", delta_df[(delta_df.domain == domain) & (delta_df.adverse)]))
    for sgname, sgdf in sgs:
        for factor in FACTORS:
            sub = sgdf[sgdf.factor == factor]
            if sub["case_id"].nunique() < MIN_SUBGROUP:
                continue
            dmean, lo, hi = cluster_bootstrap(sub, "delta_feature")
            all_rows.append({"domain": domain, "subgroup": sgname, "stage": STAGE[factor],
                             "factor": factor, "delta_feature": round(dmean, 3),
                             "ci_low": round(lo, 3), "ci_high": round(hi, 3)})
alldf = pd.DataFrame(all_rows)
save_table(alldf, "tab40_delta_all_domains")

# [Mixed-effects robustness check on case-level deltas (primary domain)]
try:
    import statsmodels.formula.api as smf
    md = delta_df[delta_df.domain == prim].copy()
    md["is_evidence"] = (md["stage"] == "Evidence").astype(int)
    model = smf.mixedlm("delta_feature ~ C(factor)", md, groups=md["case_id"])
    res = model.fit(method="lbfgs", disp=False)
    pd.DataFrame({"term": res.params.index, "coef": res.params.values,
                  "pvalue": res.pvalues.values}).to_csv(TAB_DIR / "tab51_mixed_effects.csv", index=False)
    print("mixed-effects robustness check saved")
except Exception as exc:
    print("mixed-effects step skipped:", type(exc).__name__)

print("\nGeneration + delta analysis complete.")


credentials loaded
evidence blocks: 10,500
conditions to generate: 28,000
generated 500/28,000
generated 1,000/28,000
generated 1,500/28,000
generated 2,000/28,000
generated 2,500/28,000
generated 3,000/28,000
generated 3,500/28,000
generated 4,000/28,000
generated 4,500/28,000
generated 5,000/28,000
generated 5,500/28,000
generated 6,000/28,000
generated 6,500/28,000
generated 7,000/28,000
generated 7,500/28,000
generated 8,000/28,000
generated 8,500/28,000
generated 9,000/28,000
generated 9,500/28,000
generated 10,000/28,000
generated 10,500/28,000
generated 11,000/28,000
generated 11,500/28,000
generated 12,000/28,000
generated 12,500/28,000
generated 13,000/28,000
generated 13,500/28,000
generated 14,000/28,000
generated 14,500/28,000
generated 15,000/28,000
generated 15,500/28,000
generated 16,000/28,000
generated 16,500/28,000
generated 17,000/28,000
generated 17,500/28,000
generated 18,000/28,000
generated 18,500/28,000
generated 19,000/28,000
generated 19,500/28,000
generated 2

In [2]:
# ============================================================================
# 06_generation_and_delta.ipynb  --  APPENDIX NUMBERS (run after the main cell)
# Self-contained: reads only the files the main run already wrote, so it does
# not depend on in-memory variables. Writes, for the primary domain, the
# mixed-effects variance decomposition, fixed effects, pairwise-vs-mixed
# agreement, and distance coverage -- each as a CSV and an \input-ready LaTeX
# table in results/tables/. No file path is printed.
# ============================================================================
import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
SEED = 42

ROOT = Path("..")
DATA_DIR = ROOT / "data"
TAB_DIR = ROOT / "results" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

def save_table(df, name):
    df.to_csv(TAB_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"saved table: {name}")

def save_latex(lines, name):
    (TAB_DIR / f"{name}.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"saved LaTeX table: {name}")

# ---- Config (must match the main cell) -------------------------------------
REFERENCE_MODEL, REFERENCE_XAI = "XGBoost", "SHAP"
REFERENCE_GENERATOR, REFERENCE_PROMPT = "reference", "basic"
MODELS_DISPLAY = ["XGBoost", "LightGBM", "RandomForest"]
GENERATOR_KEYS = ["weak", "reference", "strong"]
PROMPT_KEYS = ["basic", "constrained", "verification"]

# ---- Load the artefacts the main run wrote ---------------------------------
with open(DATA_DIR / "feature_registry.json", encoding="utf-8") as f:
    registry = json.load(f)
prim = next(d for d in registry if registry[d]["primary"])
corpus = pd.read_parquet(DATA_DIR / "claims_corpus.parquet")
corpus = corpus[corpus["rep"] == 0]
alldom = pd.read_csv(TAB_DIR / "tab40_delta_all_domains.csv")  # for pairwise delta

# Claims lookup: (domain, case_id, model, xai, generator, prompt) -> claim list
cidx = {}
for _, r in corpus.iterrows():
    cidx[(r["domain"], r["case_id"], r["model"], r["xai"], r["generator"], r["prompt"])] = r["claims"]
def claims_at(model, xai, gen, prompt, cid):
    return cidx.get((prim, cid, model, xai, gen, prompt))
def feat_set(cl):
    return frozenset(c["feature"] for c in cl if isinstance(c, dict) and "feature" in c)

prim_cases = sorted(corpus[corpus.domain == prim]["case_id"].unique())

# ---- Binary "was this feature mentioned?" design ---------------------------
BLOCK_CONDS = {
    "A": [(m, REFERENCE_XAI, REFERENCE_GENERATOR, REFERENCE_PROMPT) for m in MODELS_DISPLAY]
       + [(REFERENCE_MODEL, x, REFERENCE_GENERATOR, REFERENCE_PROMPT) for x in ("SHAP", "LIME")],
    "B": [(REFERENCE_MODEL, REFERENCE_XAI, g, REFERENCE_PROMPT) for g in GENERATOR_KEYS]
       + [(REFERENCE_MODEL, REFERENCE_XAI, REFERENCE_GENERATOR, p) for p in PROMPT_KEYS],
}
def mention_frame(block):
    rows = []
    for cid in prim_cases:
        colsets = {}
        for (model, xai, gen, prompt) in BLOCK_CONDS[block]:
            cl = claims_at(model, xai, gen, prompt, cid)
            colsets[(model, xai, gen, prompt)] = feat_set(cl) if cl is not None else None
        present = [s for s in colsets.values() if s is not None]
        if not present:
            continue
        candidates = set().union(*present)
        for (model, xai, gen, prompt), s in colsets.items():
            if s is None:
                continue
            for feat in candidates:
                rows.append({"case_id": cid, "feature": feat, "model": model, "xai": xai,
                             "generator": gen, "prompt": prompt, "mentioned": int(feat in s)})
    return pd.DataFrame(rows)

def resid_var(df, formula):
    return float(np.var(smf.ols(formula, data=df).fit().resid, ddof=0))

# ---- (1) Sequential variance decomposition ---------------------------------
var_rows = []
for block, factors in (("A", ["model", "xai"]), ("B", ["generator", "prompt"])):
    mf = mention_frame(block)
    if len(mf) == 0:
        continue
    base = float(np.var(mf["mentioned"], ddof=0))
    formula = "mentioned ~ C(feature)"
    prev = resid_var(mf, formula)
    var_rows.append({"block": block, "added": "feature", "residual_var": round(prev, 4),
                     "var_explained_pct": round(100 * (base - prev) / base, 2)})
    for fac in factors:
        formula += f" + C({fac})"
        cur = resid_var(mf, formula)
        var_rows.append({"block": block, "added": fac, "residual_var": round(cur, 4),
                         "var_explained_pct": round(100 * (prev - cur) / base, 2)})
        prev = cur
var_df = pd.DataFrame(var_rows)
save_table(var_df, "tab_mixed_var")
vl = ["\\begin{table}[t]", "  \\centering",
      "  \\caption{Sequential variance decomposition of the binary mention outcome,",
      "  by block (primary domain). ``Variance explained'' is the proportional",
      "  reduction in residual variance from adding the factor.}",
      "  \\label{tab:mixed-var}", "  \\small", "  \\begin{tabular}{llcc}", "    \\toprule",
      "    Block & Added factor & Residual variance & Variance explained \\\\", "    \\midrule"]
for _, r in var_df.iterrows():
    vl.append(f"    {r['block']} & {r['added']} & ${r['residual_var']:.4f}$ & ${r['var_explained_pct']:.2f}\\%$ \\\\")
vl += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
save_latex(vl, "tab_mixed_var")

# ---- (2) Fixed effects (cluster-robust by case) ----------------------------
fe_rows = []
for block, fac_a, fac_b in (("A", "model", "xai"), ("B", "generator", "prompt")):
    mf = mention_frame(block)
    if len(mf) == 0:
        continue
    res = smf.ols(f"mentioned ~ C({fac_a}) + C({fac_b})", data=mf).fit(
        cov_type="cluster", cov_kwds={"groups": mf["case_id"]})
    ci = res.conf_int()
    for term in res.params.index:
        if term == "Intercept":
            continue
        lo, hi = float(ci.loc[term][0]), float(ci.loc[term][1])
        fe_rows.append({"block": block,
                        "term": term.replace("C(", "").replace(")", "").replace("[T.", "=").rstrip("]"),
                        "coef": round(float(res.params[term]), 4),
                        "ci_low": round(lo, 4), "ci_high": round(hi, 4),
                        "excludes_zero": bool(lo > 0 or hi < 0)})
fe_df = pd.DataFrame(fe_rows)
save_table(fe_df, "tab_mixed_fe")
fl = ["\\begin{table}[t]", "  \\centering",
      "  \\caption{Fixed-effect coefficients from the linear-probability model for the",
      "  binary mention outcome, by block, with cluster-robust $95\\%$ intervals;",
      "  ${}^{\\ast}$ marks an interval excluding zero.}",
      "  \\label{tab:mixed-fe}", "  \\small", "  \\begin{tabular}{llc}", "    \\toprule",
      "    Block & Term & Coefficient \\;[95\\% CI] \\\\", "    \\midrule"]
for _, r in fe_df.iterrows():
    star = "^{\\ast}" if r["excludes_zero"] else ""
    fl.append(f"    {r['block']} & {r['term']} & ${r['coef']:.4f}$ \\;$[{r['ci_low']:.4f},\\,{r['ci_high']:.4f}]{star}$ \\\\")
fl += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
save_latex(fl, "tab_mixed_fe")

# ---- (3) Pairwise-delta vs mixed-model agreement ---------------------------
uniq = {}
for block, factors in (("A", ["model", "xai"]), ("B", ["generator", "prompt"])):
    mf = mention_frame(block)
    if len(mf) == 0:
        continue
    full = "mentioned ~ C(feature) + " + " + ".join(f"C({f})" for f in factors)
    full_rv = resid_var(mf, full)
    base = float(np.var(mf["mentioned"], ddof=0))
    for fac in factors:
        reduced = "mentioned ~ C(feature) + " + " + ".join(f"C({g})" for g in factors if g != fac)
        uniq[fac] = round((resid_var(mf, reduced) - full_rv) / base, 4)
fac_name = {"xai": "XAI method", "model": "Prediction model", "generator": "LLM", "prompt": "Prompt"}
prim_all = alldom[(alldom.domain == prim) & (alldom.subgroup == "all")].set_index("factor")["delta_feature"]
ag_rows = []
for f in uniq:
    name = fac_name[f]
    ag_rows.append({"factor": name,
                    "pairwise_delta": float(prim_all.get(name, np.nan)),
                    "unique_var_contrib": uniq[f]})
ag = pd.DataFrame(ag_rows)
ag["pairwise_rank"] = ag["pairwise_delta"].rank(ascending=False)
ag["unique_rank"] = ag["unique_var_contrib"].rank(ascending=False)
agree_rho = (float(spearmanr(ag["pairwise_delta"], ag["unique_var_contrib"]).correlation)
             if ag["pairwise_delta"].notna().sum() > 2 else float("nan"))
save_table(ag, "tab_mixed_agree")
al = ["\\begin{table}[t]", "  \\centering",
      "  \\caption{The two analytical routes compared (primary domain). The pairwise",
      "  $\\delta$ is the within-case additional variation (feature distance); the unique",
      "  contribution is the leave-one-out residual-variance reduction from the mixed",
      f"  model. Spearman $\\rho = {agree_rho:.2f}$ between the two rankings.}}",
      "  \\label{tab:mixed-agree}", "  \\small", "  \\begin{tabular}{lcc}", "    \\toprule",
      "    Factor & Pairwise $\\delta$ (rank) & Unique contrib.\\ (rank) \\\\", "    \\midrule"]
for _, r in ag.iterrows():
    al.append(f"    {r['factor']} & ${r['pairwise_delta']:.3f}$ \\;({int(r['pairwise_rank'])}) & "
              f"${r['unique_var_contrib']:.4f}$ \\;({int(r['unique_rank'])}) \\\\")
al += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
save_latex(al, "tab_mixed_agree")

# ---- (4) Distance coverage: undefined direction/rank pairs by factor -------
FACTORS = {
    "XAI method": (lambda cid: (claims_at("XGBoost", "SHAP", REFERENCE_GENERATOR, REFERENCE_PROMPT, cid),
                                claims_at("XGBoost", "LIME", REFERENCE_GENERATOR, REFERENCE_PROMPT, cid))),
    "Prediction model": (lambda cid: (claims_at("XGBoost", "SHAP", REFERENCE_GENERATOR, REFERENCE_PROMPT, cid),
                                       claims_at("LightGBM", "SHAP", REFERENCE_GENERATOR, REFERENCE_PROMPT, cid))),
    "LLM": (lambda cid: (claims_at(REFERENCE_MODEL, REFERENCE_XAI, "reference", REFERENCE_PROMPT, cid),
                         claims_at(REFERENCE_MODEL, REFERENCE_XAI, "strong", REFERENCE_PROMPT, cid))),
    "Prompt": (lambda cid: (claims_at(REFERENCE_MODEL, REFERENCE_XAI, REFERENCE_GENERATOR, "basic", cid),
                            claims_at(REFERENCE_MODEL, REFERENCE_XAI, REFERENCE_GENERATOR, "constrained", cid))),
}
STAGE = {"XAI method": "Evidence", "Prediction model": "Evidence", "LLM": "Generation", "Prompt": "Generation"}
cov_rows = []
for factor, getter in FACTORS.items():
    n = und_dir = und_rank = 0
    for cid in prim_cases:
        a, b = getter(cid)
        if a is None or b is None:
            continue
        shared = len(feat_set(a) & feat_set(b))
        n += 1
        und_dir += int(shared < 1)
        und_rank += int(shared < 2)
    if n:
        cov_rows.append({"factor": factor, "stage": STAGE[factor], "n_pairs": n,
                         "direction_undefined": round(und_dir / n, 3),
                         "rank_undefined": round(und_rank / n, 3)})
cov_df = pd.DataFrame(cov_rows)
save_table(cov_df, "tab_coverage")
cl2 = ["\\begin{table}[t]", "  \\centering",
       "  \\caption{Share of same-case pairs for which the direction and rank distances",
       "  are undefined (fewer than one, resp.\\ two, shared drivers), by the component",
       "  varied (primary domain). Feature distance is always defined.}",
       "  \\label{tab:coverage}", "  \\small", "  \\begin{tabular}{llccc}", "    \\toprule",
       "    Component & Stage & Pairs & Direction undef.\\ & Rank undef. \\\\", "    \\midrule"]
for _, r in cov_df.iterrows():
    cl2.append(f"    {r['factor']} & {r['stage']} & {int(r['n_pairs'])} & "
               f"${r['direction_undefined']:.3f}$ & ${r['rank_undefined']:.3f}$ \\\\")
cl2 += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
save_latex(cl2, "tab_coverage")

print("\nAppendix tables written. Variance decomposition:")
print(var_df.to_string(index=False))
print("\nCoverage:")
print(cov_df.to_string(index=False))

saved table: tab_mixed_var
saved LaTeX table: tab_mixed_var
saved table: tab_mixed_fe
saved LaTeX table: tab_mixed_fe
saved table: tab_mixed_agree
saved LaTeX table: tab_mixed_agree
saved table: tab_coverage
saved LaTeX table: tab_coverage

Appendix tables written. Variance decomposition:
block     added  residual_var  var_explained_pct
    A   feature        0.2142              11.92
    A     model        0.2142               0.00
    A       xai        0.2142               0.00
    B   feature        0.0321              15.14
    B generator        0.0315               1.80
    B    prompt        0.0314               0.24

Coverage:
          factor      stage  n_pairs  direction_undefined  rank_undefined
      XAI method   Evidence      297                0.003           0.071
Prediction model   Evidence      297                0.000           0.003
             LLM Generation      297                0.000           0.000
          Prompt Generation      297                0.000   